In [53]:
from paths import *
import os
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import pandas as pd

In [54]:
# Configs
envS_path = os.path.join(model_save_, f"eval-C_0Tj-1021123825")
envP_path = os.path.join(model_save_, f"eval-C_0Tk-1021183234")
task_name = "ABXSomethingPro-stop-data-aspirationRangeComp"
range_start, range_end = 2, 4
model_condition = "b"
strseq_learned_runs = "12345"


In [55]:
# Collect data
envS_res = {}
envP_res = {}
for hiddim in [4, 8, 16, 32, 48, 64]: 
    model_type = f"recon{hiddim}-phi"
    for environment, env_res, res_save_dir in zip(["S", "P"], [envS_res, envP_res], [envS_path, envP_path]):
        layered_res = {}
        look_for_layer_path = f"{task_name}-{range_start}-{range_end}"
        # Read ori
        ori_path = os.path.join(res_save_dir, look_for_layer_path, f"07-save-ari-recon64-phi-{model_condition}-{strseq_learned_runs}-ori.npy")
        if os.path.exists(ori_path): 
            ori_res = np.load(ori_path)
        else: 
            raise ValueError("No ori path found.")
        
        for layer in ["hidrep", "attnout"]: # "enc-lin1", 
            """, "dec-lin1", "enc-lin1", 
                        "dec-rnn1-f", "enc-rnn1-f", "enc-rnn1-b",
                        "dec-rnn2-f", "enc-rnn2-f", "enc-rnn2-b", 
                        "dec-rnn3-f", "enc-rnn3-f", "enc-rnn3-b", 
                        "dec-rnn4-f", "enc-rnn4-f", "enc-rnn4-b", 
                        "dec-rnn5-f", "enc-rnn5-f", "enc-rnn5-b", """
            # print(f"Processing {model_type} in layer {layer}...")
            asp_list_epochs = []
            layer_path = os.path.join(res_save_dir, look_for_layer_path, 
                                        f"07-save-ari-{model_type}-{model_condition}-{strseq_learned_runs}-{layer}.npy")
            if os.path.exists(layer_path): 
                layer_res = np.load(layer_path)
            else: 
                print(f"Warning: {layer_path} not found. ")
                layer_res = np.zeros_like(ori_res)
            layered_res[layer] = layer_res
        layered_res["ori"] = ori_res
        env_res[hiddim] = layered_res


In [56]:
# Define test functions
def test_arrays(data1, data2, test_epoch_range=(0, 101)): 
    # Assuming `data1` and `data2` are your two numpy arrays of shape (num_runs, num_epochs)
    # Choose your test (paired t-test or Wilcoxon signed-rank test)
    p_values = {}
    t_stats = {}

    # Loop over each epoch to perform the significance test
    for epoch in test_epoch_range:
        t_stats[epoch], p_values[epoch] = ttest_rel(data1[:, epoch], data2[:, epoch])
    return t_stats, p_values

def test_arrays_merged(data1, data2, test_fun, test_epoch_range=(0, 101)): 
    # Assuming `data1` and `data2` are your two numpy arrays of shape (num_runs, num_epochs)
    # Choose your test (paired t-test or Wilcoxon signed-rank test)

    data1_pooled = data1[:, test_epoch_range[0]:test_epoch_range[1]].flatten()
    data2_pooled = data2[:, test_epoch_range[0]:test_epoch_range[1]].flatten()

    # return ttest_rel(data1_pooled, data2_pooled)
    return test_fun(data1_pooled, data2_pooled)

## Main Running Here

In [ ]:
thelayer = "attnout"
# thelayer = "hidrep"
# test_fun = ttest_rel
test_fun = wilcoxon
test_epoch_range = range(60, 80)
for test_hiddim in [4, 8, 16, 32, 48, 64]: 
    envS_ori_attn_diff = envS_res[test_hiddim][thelayer]-envS_res[test_hiddim]["ori"]
    envP_ori_attn_diff = envP_res[test_hiddim][thelayer]-envP_res[test_hiddim]["ori"]
    print(f"Testing hiddim={test_hiddim}: S/P, S-a/o, P-a/o, S-P")
    _, p_values = test_arrays_merged(envS_ori_attn_diff, envP_ori_attn_diff, test_fun, test_epoch_range)
    _, p_values2 = test_arrays_merged(envS_res[test_hiddim][thelayer], envS_res[test_hiddim]["ori"], test_fun, test_epoch_range)
    _, p_values3 = test_arrays_merged(envP_res[test_hiddim][thelayer], envP_res[test_hiddim]["ori"], test_fun, test_epoch_range)
    mean_diff = (envS_ori_attn_diff[:, test_epoch_range[0]:test_epoch_range[1]] - envP_ori_attn_diff[:, test_epoch_range[0]:test_epoch_range[1]]).mean()
    
    print(p_values, p_values2, p_values3, mean_diff)

Testing hiddim=4: S/P, S-a/o, P-a/o, S-P
1.2938800168252932e-17 3.7369878171040194e-15 4.2663227307182344e-18 -0.2697717105263157
Testing hiddim=8: S/P, S-a/o, P-a/o, S-P
3.2823321022861196e-14 2.780026650061847e-06 1.13488682647525e-15 -0.09872697368421052
Testing hiddim=16: S/P, S-a/o, P-a/o, S-P
6.606662752812207e-10 4.312288022089701e-10 1.0436051216112079e-05 -0.08066447368421052
Testing hiddim=32: S/P, S-a/o, P-a/o, S-P
0.0025087212297265674 1.360775369354392e-09 0.010523939274295915 -0.028975657894736835
Testing hiddim=48: S/P, S-a/o, P-a/o, S-P
0.06737132717421686 8.891709634750892e-11 6.050955668364737e-05 -0.016830263157894723
Testing hiddim=64: S/P, S-a/o, P-a/o, S-P
0.5314624520873817 3.22372750749634e-08 9.220831967350151e-08 -0.006156578947368414


In [51]:
thelayer = "attnout"
# thelayer = "hidrep"
# test_fun = ttest_rel
test_fun = wilcoxon
for i in range(0, 100): 
    # test_epoch_range = range(40, 101)
    test_epoch_range = range(i, i+2)
    for test_hiddim in [32, 48, 64]: 
        envS_ori_attn_diff = envS_res[test_hiddim][thelayer]/envS_res[test_hiddim]["ori"]
        envP_ori_attn_diff = envP_res[test_hiddim][thelayer]/envP_res[test_hiddim]["ori"]
        # print(f"Testing hiddim={test_hiddim}: S/P, S-a/o, P-a/o, S-P")
        _, p_values = test_arrays_merged(envS_ori_attn_diff, envP_ori_attn_diff, test_fun, test_epoch_range)
        # _, p_values2 = test_arrays_merged(envS_res[test_hiddim][thelayer], envS_res[test_hiddim]["ori"], test_fun, test_epoch_range)
        # _, p_values3 = test_arrays_merged(envP_res[test_hiddim][thelayer], envP_res[test_hiddim]["ori"], test_fun, test_epoch_range)
        # mean_diff = (envS_ori_attn_diff[:, test_epoch_range[0]:test_epoch_range[1]] - envP_ori_attn_diff[:, test_epoch_range[0]:test_epoch_range[1]]).mean()
        # print(p_values, p_values2, p_values3, mean_diff)
        print("*" if p_values <= 0.05 else "-", end=", ")
    print()

*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, *, 
*, *, 

*, 
*, *, *, 
*, *, *, 
*, *, -, 
*, *, *, 
*, *, -, 
*, *, -, 
*, *, -, 
*, *, -, 
*, *, *, 
*, *, *, 
*, *, *, 
*, -, -, 
*, *, -, 
*, -, -, 
*, -, -, 
*, *, -, 
*, *, -, 
*, *, -, 
*, -, -, 
-, -, -, 
*, *, *, 
-, -, -, 
*, *, *, 
*, *, -, 
*, *, *, 
*, -, -, 
*, *, -, 
-, *, -, 
-, -, *, 
*, -, -, 
*, *, -, 
*, -, -, 
-, *, -, 
-, -, -, 
*, -, -, 
*, -, -, 
*, *, -, 
*, -, -, 
*, -, -, 
*, -, -, 
*, -, -, 
-, *, -, 
*, -, -, 
*, *, -, 
*, -, -, 
-, -, -, 
-, -, -, 
*, -, -, 
*, *, -, 
*, *, -, 
*, *, -, 
*, *, -, 
*, -, -, 
-, *, -, 
-, *, -, 
*, *, -, 
-, -, -, 
*, *, *, 
-, -, -, 
*, *, -, 
-, -, -, 
*, *, -, 
*, -, -, 
*, *, *, 
*, *, -, 
*, *, -, 
-, *, -, 
-, *, -, 
*, *, -, 
-, *, -, 
-, *, *, 
-, -, -, 
-, -, -, 
-, -, -, 
-, *, -, 
*, -, -, 
-, *, -, 
-, -, -, 
*, *, -, 
-, *, -, 
-, *, *, 
-, *, -, 


In [150]:
def dfify(array): 
    num_runs, num_epoch = array.shape
    values = array.flatten()
    # Create 'epoch' and 'run' columns using broadcasting
    runs = np.repeat(np.arange(num_runs), num_epoch)
    epochs = np.tile(np.arange(num_epoch), num_runs)
    df = pd.DataFrame({
        'epoch': epochs,
        'run': runs,
        'value': values
    })
    return df

In [154]:
test_hiddim = 32
envS_ori_attn_diff = envS_res[test_hiddim][thelayer]/envS_res[test_hiddim]["ori"]
envP_ori_attn_diff = envP_res[test_hiddim][thelayer]/envP_res[test_hiddim]["ori"]

epoch_range = (40, 101)
num_epoch = len(epoch_range)
array1 = envS_ori_attn_diff[:, epoch_range[0]:epoch_range[1]]
array2 = envS_ori_attn_diff[:, epoch_range[0]:epoch_range[1]]

df1 = dfify(array1)
df2 = dfify(array2)

# Add condition and run identifiers
df1['condition'] = 'S'
df2['condition'] = 'P'

# # Combine the two DataFrames
df = pd.concat([df1, df2]).reset_index(drop=True)


In [155]:
df

,epoch,run,value,condition
0,0,0,0.793214,S
1,1,0,0.917315,S
2,2,0,1.096471,S
3,3,0,0.875000,S
4,4,0,1.002516,S
...,...,...,...,...
12195,56,99,0.965487,P
12196,57,99,0.831948,P
12197,58,99,1.023226,P
12198,59,99,1.011234,P


In [158]:
df.to_csv("jkres.csv")

In [157]:
import statsmodels.formula.api as smf

# Fit the mixed-effects model
# Example formula: 'value ~ condition + epoch' with random intercept by 'run'
model = smf.mixedlm("value ~ condition + epoch", df, groups=df["run"])
result = model.fit()

# Print the model summary
print(result.summary())


          Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  value    
No. Observations:  12200    Method:              REML     
No. Groups:        100      Scale:               0.0278   
Min. group size:   122      Log-Likelihood:      4436.8677
Max. group size:   122      Converged:           Yes      
Mean group size:   122.0                                  
----------------------------------------------------------
               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept       0.909    0.005 189.208 0.000  0.899  0.918
condition[T.S]  0.000    0.003   0.000 1.000 -0.006  0.006
epoch          -0.000    0.000  -0.103 0.918 -0.000  0.000
Group Var       0.001    0.001                            



/home/franklhtan/anaconda3/envs/wavln/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## Other codes

In [7]:
# Shape of each layer's result: (num_runs, num_epochs)
test_hiddim = 32
test_epoch_range = range(85, 101)
envS_ori_attn_diff = envS_res[test_hiddim]["attnout"] - envS_res[test_hiddim]["ori"]
envP_ori_attn_diff = envP_res[test_hiddim]["attnout"] - envP_res[test_hiddim]["ori"]

envS_ori_attn_diff_mag = envS_ori_attn_diff * 1
envP_ori_attn_diff_mag = envP_ori_attn_diff * 1
# Shape: (num_runs, num_epochs)

t_stats, p_values = test_arrays(envS_ori_attn_diff_mag, envP_ori_attn_diff_mag, test_epoch_range)
merged_t_stat, merged_p_value = test_arrays_merged(envS_ori_attn_diff_mag, envP_ori_attn_diff_mag, test_epoch_range)


In [8]:
merged_t_stat, merged_p_value

(-1.534549540683356, 0.12808424948257804)

In [10]:
envS_ori_attn_diff_mag[:, 95]

array([-0.02414474, -0.02730263, -0.14      ,  0.01006579, -0.04993421,
       -0.03401316, -0.08907895, -0.13631579, -0.03565789, -0.07151316,
       -0.04861842, -0.08657895,  0.03388158,  0.09118421, -0.03328947,
       -0.03447368,  0.12782895, -0.04190789, -0.13559211, -0.00684211,
       -0.00756579,  0.06901316,  0.03322368,  0.05026316, -0.00684211,
       -0.065     ,  0.06875   , -0.11493421,  0.09690789, -0.05263158,
        0.01717105,  0.06355263, -0.02559211,  0.10703947, -0.15888158,
       -0.04572368,  0.05342105,  0.11552632,  0.04203947, -0.06769737,
        0.11098684, -0.01414474,  0.02888158, -0.04631579,  0.02743421,
       -0.07861842, -0.02986842, -0.07203947,  0.04868421, -0.15375   ,
       -0.07355263,  0.04460526, -0.03394737,  0.00289474, -0.07782895,
       -0.10815789,  0.02447368, -0.00118421,  0.05677632,  0.01368421,
        0.06565789, -0.04848684,  0.04375   , -0.10151316,  0.22888158,
        0.00335526,  0.01796053, -0.04960526, -0.06006579, -0.10

In [9]:
t_stats, p_values

({85: -1.534549540683356,
  86: -2.7962242031594675,
  87: -1.3355221499088714,
  88: -0.7009500995549316,
  89: 0.8371594510877548,
  90: -0.3187771531634663,
  91: -0.9156960202871902,
  92: -0.8034461847568874,
  93: -2.481061936998119,
  94: -1.7435020547640396,
  95: 0.5325607477290834,
  96: -4.281139259043023,
  97: -1.911353335624819,
  98: -2.152103940354532,
  99: -0.42396561381876374,
  100: -3.0639826042726757},
 {85: 0.12808424948257804,
  86: 0.006211876398403946,
  87: 0.1847679448493376,
  88: 0.48497844347774366,
  89: 0.404519311525704,
  90: 0.7505670828813575,
  91: 0.3620514531196085,
  92: 0.4236417605873828,
  93: 0.014784728116952828,
  94: 0.08435035174732526,
  95: 0.5955310021750793,
  96: 4.304593138232808e-05,
  97: 0.05885139862272823,
  98: 0.03382040025916915,
  99: 0.6725106838241294,
  100: 0.00281400554026946})

In [126]:
from statsmodels.stats.power import TTestPower

# Parameters
sample_size = 1000      # Sample size per group (assuming paired samples, n = 30 in total)
alpha = 0.05          # Significance threshold
power = 0.8           # Desired power (often 0.8)

# Initialize power analysis object for a paired t-test
power_analysis = TTestPower()

# Calculate effect size
effect_size = power_analysis.solve_power(nobs=sample_size, alpha=alpha, power=power, alternative='two-sided')
print("Minimum detectable effect size:", effect_size)


Minimum detectable effect size: 0.08867912597397398


In [62]:
# Shape of each layer's result: (num_runs, num_epochs)
test_hiddim = 32
test_epoch_range = range(40, 101)
envS_ori_attn_diff = envS_res[test_hiddim]["attnout"]/envS_res[test_hiddim]["ori"]
envP_ori_attn_diff = envP_res[test_hiddim]["attnout"]/envP_res[test_hiddim]["ori"]

envS_ori_attn_diff_mag = envS_ori_attn_diff * 1
envP_ori_attn_diff_mag = envP_ori_attn_diff * 1
# Shape: (num_runs, num_epochs)

# t_stats, p_values = test_arrays(envS_ori_attn_diff_mag, envP_ori_attn_diff_mag, test_epoch_range)


In [63]:
test_arrays_merged(envS_ori_attn_diff_mag, envP_ori_attn_diff_mag, ttest_rel, test_epoch_range), test_arrays_merged(envS_ori_attn_diff_mag, envP_ori_attn_diff_mag, wilcoxon, test_epoch_range)

(TtestResult(statistic=-1.6632544183515092, pvalue=0.09942422018667348, df=99),
 WilcoxonResult(statistic=1936.0, pvalue=0.04284956892471004))

In [66]:
test_arrays_merged(envS_res[test_hiddim]["attnout"], envS_res[test_hiddim]["ori"], wilcoxon, test_epoch_range)

WilcoxonResult(statistic=1250.5, pvalue=1.175078102982852e-05)

In [67]:
test_arrays_merged(envP_res[test_hiddim]["attnout"], envP_res[test_hiddim]["ori"], wilcoxon, test_epoch_range)

WilcoxonResult(statistic=1958.5, pvalue=0.05143723469282332)

In [68]:
test_arrays_merged(envP_res[test_hiddim]["attnout"], envS_res[test_hiddim]["attnout"], wilcoxon, test_epoch_range)

WilcoxonResult(statistic=2194.5, pvalue=0.32756662499461175)